# Step 5: Practice — Explore Federated Learning
**TSU NSF AI Workshop 2026 — Federated Learning Lab**

---

## Your Turn!

In this notebook, you will **experiment** with federated learning by changing key parameters.
There is no single "right" answer — the goal is to **observe, think, and discuss**.

### How to use this notebook
1. Find the **"CHANGE ME"** cell below
2. Modify one parameter at a time
3. Run all cells (`Runtime > Run all`)
4. Record the final accuracy in the table below
5. Discuss your findings with your group

### Record your results here (edit this cell!)

| Experiment | NUM_ROUNDS | LOCAL_EPOCHS | LEARNING_RATE | NON_IID_DEGREE | Final Accuracy |
|------------|-----------|-------------|--------------|---------------|---------------|
| Default    | 5         | 2           | 0.001        | 0.8           | ??% |
| Exp 1      |           |             |              |               | |
| Exp 2      |           |             |              |               | |
| Exp 3      |           |             |              |               | |
| Exp 4      |           |             |              |               | |

## Parameter Guide

### `NUM_ROUNDS` — Communication rounds
How many times do the clients and server exchange model weights?
- **Try:** `1`, `3`, `5`, `10`
- More rounds = higher accuracy, but higher communication cost
- Is there a point where adding more rounds stops helping much?

### `LOCAL_EPOCHS` — Local training per round
How many epochs does each client train before sending weights back?
- **Try:** `1`, `2`, `5`, `10`
- More local epochs = less communication, but risk of **client drift**
- *Client drift:* each client's model diverges in different directions — averaging them becomes less effective

### `LEARNING_RATE` — Step size for learning
How large are the weight updates during training?
- **Try:** `0.01`, `0.001`, `0.0001`
- Too high → training is unstable (accuracy jumps around)
- Too low → training is very slow

### `NON_IID_DEGREE` — How different are the clients?
Controls how skewed the data split is between Client 1 and Client 2.
- **Try:** `0.5` (balanced / IID), `0.7`, `0.8`, `0.9` (very skewed)
- `0.5` = both clients have a roughly equal mix of all classes → easiest for FL
- `0.9` = clients have very different diseases → hardest for FL

## ✏️ Change These Parameters!

In [ ]:
# ============================================================
# CHANGE THESE PARAMETERS AND RE-RUN THE NOTEBOOK
# ============================================================

NUM_ROUNDS     = 5      # Try: 1, 3, 5, 10
LOCAL_EPOCHS   = 2      # Try: 1, 2, 5, 10
LEARNING_RATE  = 0.001  # Try: 0.01, 0.001, 0.0001
NON_IID_DEGREE = 0.8    # Try: 0.5 (balanced) to 0.9 (very skewed)

# ============================================================

print("Your experiment settings:")
print(f"  NUM_ROUNDS     = {NUM_ROUNDS}")
print(f"  LOCAL_EPOCHS   = {LOCAL_EPOCHS}")
print(f"  LEARNING_RATE  = {LEARNING_RATE}")
print(f"  NON_IID_DEGREE = {NON_IID_DEGREE}  (0.5=balanced IID, 0.9=very non-IID)")

## Setup (do not change below this line)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

DRIVE_BASE = "/content/drive/MyDrive/FederatedLearning"
DATA_PATH  = os.path.join(DRIVE_BASE, "data", "dermamnist.npz")
NUM_CLASSES = 7
BATCH_SIZE  = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def make_loader(images, labels, shuffle=True):
    x = torch.tensor(images / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
    y = torch.tensor(labels, dtype=torch.long)
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=shuffle)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(1) == lbls).sum().item()
            total   += lbls.size(0)
    return correct / total

def client_update(global_model, loader, local_epochs, lr):
    local_model = copy.deepcopy(global_model)
    local_model.train()
    opt = optim.Adam(local_model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(local_epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            crit(local_model(imgs), lbls).backward()
            opt.step()
    return local_model.state_dict()

def federated_average(global_model, client_weights, client_sizes):
    total = sum(client_sizes)
    avg   = copy.deepcopy(client_weights[0])
    for key in avg:
        avg[key] = torch.zeros_like(avg[key], dtype=torch.float32)
    for weights, n in zip(client_weights, client_sizes):
        for key in avg:
            avg[key] += weights[key].float() * (n / total)
    global_model.load_state_dict(avg)
    return global_model

print("Helpers ready.")

## Load & Split Data Using Your NON_IID_DEGREE

The chart below shows how different the two clients' class distributions are given your setting.

In [ ]:
CLASS_NAMES = ["Actinic ker.", "Basal cell", "Benign ker.",
               "Dermatofibroma", "Melanoma", "Melanocytic nevi", "Vascular"]

data = np.load(DATA_PATH)
train_images = data["train_images"]
train_labels = data["train_labels"].flatten()

np.random.seed(42)
client1_idx, client2_idx = [], []
for c in range(NUM_CLASSES):
    idx = np.where(train_labels == c)[0]
    np.random.shuffle(idx)
    # Rare classes (0-3): Client 1 gets NON_IID_DEGREE fraction
    # Common classes (4-6): Client 1 gets (1 - NON_IID_DEGREE) fraction
    frac = NON_IID_DEGREE if c <= 3 else (1 - NON_IID_DEGREE)
    split = int(len(idx) * frac)
    client1_idx.extend(idx[:split])
    client2_idx.extend(idx[split:])

client1_idx = np.array(client1_idx)
client2_idx = np.array(client2_idx)
c1_labels   = train_labels[client1_idx]
c2_labels   = train_labels[client2_idx]

# Visualize the split
x = np.arange(NUM_CLASSES)
n1 = [int(np.sum(c1_labels == c)) for c in range(NUM_CLASSES)]
n2 = [int(np.sum(c2_labels == c)) for c in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, n1, width=0.4, label=f"Client 1", color="steelblue")
ax.bar(x + 0.2, n2, width=0.4, label=f"Client 2", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=8)
ax.set_title(f"Data Split with NON_IID_DEGREE = {NON_IID_DEGREE}  "
             f"({'balanced' if NON_IID_DEGREE == 0.5 else 'skewed'})", fontsize=11)
ax.set_ylabel("Number of samples")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Client 1: {len(client1_idx)} samples | Client 2: {len(client2_idx)} samples")
if NON_IID_DEGREE == 0.5:
    print("→ IID split: both clients have a similar mix of all classes.")
elif NON_IID_DEGREE >= 0.8:
    print("→ Strong non-IID: clients see very different disease distributions.")

client1_loader = make_loader(train_images[client1_idx], c1_labels)
client2_loader = make_loader(train_images[client2_idx], c2_labels)
test_loader    = make_loader(data["test_images"], data["test_labels"].flatten(), shuffle=False)
client_sizes   = [len(client1_idx), len(client2_idx)]

## Run Federated Training with Your Settings

In [ ]:
print(f"Running FL: {NUM_ROUNDS} rounds, {LOCAL_EPOCHS} local epochs, lr={LEARNING_RATE}\n")

global_model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
round_accs   = []

for rnd in range(1, NUM_ROUNDS + 1):
    w1 = client_update(global_model, client1_loader, LOCAL_EPOCHS, LEARNING_RATE)
    w2 = client_update(global_model, client2_loader, LOCAL_EPOCHS, LEARNING_RATE)
    global_model = federated_average(global_model, [w1, w2], client_sizes)
    acc = evaluate(global_model, test_loader)
    round_accs.append(acc)
    bar = "█" * int(acc * 40)
    print(f"  Round {rnd:2d}/{NUM_ROUNDS} | Accuracy: {acc:.2%}  {bar}")

print(f"\nFinal Test Accuracy: {round_accs[-1]:.2%}")

## Your Results

In [ ]:
MODEL_SIZE_MB = 2.0
total_comm = NUM_ROUNDS * 2 * 2 * MODEL_SIZE_MB

# Plot convergence curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, NUM_ROUNDS + 1), round_accs, marker="o", color="seagreen", linewidth=2)
for i, acc in enumerate(round_accs):
    ax.annotate(f"{acc:.1%}", xy=(i+1, acc), xytext=(0, 8),
                textcoords="offset points", ha="center", fontsize=8)
ax.set_xlabel("Communication Round")
ax.set_ylabel("Test Accuracy")
ax.set_title(f"Federated Learning: NON_IID={NON_IID_DEGREE}, "
             f"Rounds={NUM_ROUNDS}, LocalEpochs={LOCAL_EPOCHS}, LR={LEARNING_RATE}")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 50)
print("   EXPERIMENT SUMMARY")
print("=" * 50)
print(f"   NUM_ROUNDS     = {NUM_ROUNDS}")
print(f"   LOCAL_EPOCHS   = {LOCAL_EPOCHS}")
print(f"   LEARNING_RATE  = {LEARNING_RATE}")
print(f"   NON_IID_DEGREE = {NON_IID_DEGREE}")
print(f"   Final Accuracy = {round_accs[-1]:.2%}")
print(f"   Comm. Cost     = {total_comm:.1f} MB total")
print("=" * 50)

## Discussion Questions

Discuss these with your group. Edit this cell to write your answers!

---

**Q1. More rounds = better accuracy?**
Run with `NUM_ROUNDS = 1` then `NUM_ROUNDS = 10`.
At what point did adding more rounds stop helping? Why do you think that is?

*Your answer:*

---

**Q2. Too much local training (client drift)?**
Run with `LOCAL_EPOCHS = 1` then `LOCAL_EPOCHS = 10`.
What happened to the final accuracy? Why might training MORE locally sometimes hurt the global model?

*Your answer:*

---

**Q3. Non-IID vs IID — does it matter?**
Run with `NON_IID_DEGREE = 0.5` (balanced) then `NON_IID_DEGREE = 0.9` (very skewed).
How much did the accuracy change? Why is non-IID data a challenge for federated learning?

*Your answer:*

---

**Q4. Budget constraint — if you can only use 20 MB total communication, what is your best configuration?**
20 MB ÷ (2 clients × 2 directions × 2 MB/model) = 2.5 → you can afford **2 rounds**.
How would you set `LOCAL_EPOCHS` to maximize accuracy with only 2 rounds?

*Your answer:*

---

**Q5. (Advanced) What privacy risks remain?**
Even though raw images are not shared, is federated learning 100% private?
Research "model inversion attacks" or "gradient leakage" — what do you find?

*Your answer:*